In [1]:
import json
import pandas as pd

In [6]:

test_file = r'\\10.1.19.245\C2M Logistics\BI Analytics\Plan\План 2025 объёмы for db.xlsx'
df = pd.read_excel(test_file, sheet_name=0, engine='openpyxl')
print(df.head())

  country service  wq   tariff_range    Value       Date  Linehall
0      UZ      RM  кг  A,B,C,D,E,F,G  4058.22 2025-01-01  Лайнхолл
1      UZ      RM  кг  A,B,C,D,E,F,G  4387.74 2025-02-01  Лайнхолл
2      UZ      RM  кг  A,B,C,D,E,F,G  4649.76 2025-03-01  Лайнхолл
3      UZ      RM  кг  A,B,C,D,E,F,G  3920.11 2025-04-01  Лайнхолл
4      UZ      RM  кг  A,B,C,D,E,F,G  3542.96 2025-05-01  Лайнхолл


In [29]:
columns_to_read = {item['originalName']: str for item in schema }
print(columns_to_read)

{'itemid': <class 'str'>, 'cartonid': <class 'str'>, 'masterno': <class 'str'>, 'ordercode': <class 'str'>, 'PackageWeight(g)': <class 'str'>, 'consigneecountry': <class 'str'>, 'service': <class 'str'>}


In [40]:
dtype_map = {}
rename_map = {}
for item in schema:
    original_name = item['originalName']
    db_name = item['db_name']
    datatype = item['datatype']
    
    dtype_map[original_name] = datatype
    rename_map[original_name] = db_name
print(rename_map)

{'itemid': 'itemid', 'cartonid': 'cartonid', 'masterno': 'masterno', 'ordercode': 'ordercode', 'PackageWeight(g)': 'weight_g', 'consigneecountry': 'country', 'service': 'service'}


In [41]:
df = pd.read_excel(test_file, sheet_name=0, engine='openpyxl', usecols=columns_to_read, dtype=dtype_map)
print(df.head())

        cartonid         itemid     masterno         ordercode  \
0  DM00026714BIG  RX110267845UZ  18015090740  LP00622253065565   
1  DM00026237BIG  SX110186611UZ  18015090740  LP00621490775561   
2  DM00026237BIG  UX110641911UZ  18015090740  LP00621443573354   
3  DM00026237BIG  UX110708803UZ  18015090740  LP00621602111888   
4  DM00026237BIG  UX110662085UZ  18015090740  LP00621874718056   

   PackageWeight(g) consigneecountry service  
0                 4               UZ      RM  
1                13               UZ     NRM  
2                56               KZ     NRM  
3                63               KZ     NRM  
4                22               AZ     NRM  


In [42]:

df.rename(columns=rename_map, inplace=True)
print(df.head())

        cartonid         itemid     masterno         ordercode  weight_g  \
0  DM00026714BIG  RX110267845UZ  18015090740  LP00622253065565         4   
1  DM00026237BIG  SX110186611UZ  18015090740  LP00621490775561        13   
2  DM00026237BIG  UX110641911UZ  18015090740  LP00621443573354        56   
3  DM00026237BIG  UX110708803UZ  18015090740  LP00621602111888        63   
4  DM00026237BIG  UX110662085UZ  18015090740  LP00621874718056        22   

  country service  
0      UZ      RM  
1      UZ     NRM  
2      KZ     NRM  
3      KZ     NRM  
4      AZ     NRM  


In [45]:
import re
input_string = r"\\10.1.19.245\C2M Logistics\2024\CAINIAO\1. Январь\C2M-207, 784-04764012 (03.01.2024)- 7 days\Manifest\Manifest_784-04764012.xlsx"
pattern = r'-(\d+)[,\s]'
match = re.search(pattern, input_string)

if match:
    print(match.group(1))
else:
    print("Номер не найден")

207


In [44]:
import re

# Исходная строка
input_string = r"\\10.1.19.245\C2M Logistics\2024\CAINIAO\1. Январь\C2M-231, 180-15090843 (28.01.2024)- Kropp\Manifest\Manifest_180-15090843.xlsx"

# Регулярное выражение для поиска даты
pattern = r'\((\d{2}\.\d{2}\.\d{4})\)'

# Поиск совпадения
match = re.search(pattern, input_string)

# Если совпадение найдено, выводим дату
if match:
    print(match.group(1))
else:
    print("Дата не найдена")

28.01.2024


In [47]:
import re
from datetime import datetime

# Исходная строка
file_path = r"\\10.1.19.245\C2M Logistics\2024\CAINIAO\3. Март\C2M-263, 250-51157514 (03.03.2024)- Kropp\Manifest\Manifest_250-51157514.xlsx"

# Регулярное выражение для поиска номера и даты
PACKAGE_PATTERN = re.compile(r"-\s*(\d+).*?\((\d{2}\.\d{2}\.\d{4})\)")

# Поиск совпадения
match = PACKAGE_PATTERN.search(file_path)

if match:
    package_num = match.group(1)
    dt_package = datetime.strptime(match.group(2), '%d.%m.%Y').strftime('%Y-%m-%d')
    
    print(f"Номер: {package_num}")
    print(f"Дата: {dt_package}")
else:
    print("Совпадений не найдено")

Номер: 263
Дата: 2024-03-03


In [1]:
import psycopg2


In [18]:
conn_string = """
    host=dmprodpostgres2dzc6c48-4597z1.h.ae-rus.net,dmprodpostgres2dzc6c48-4611z4.h.ae-rus.net
    port=6532
    dbname=dm-hub-service-package-external
    user=dm-hub-service-package-external-read
    password=3QiPQjwquoINHjMY
"""

In [19]:
# Инициализация переменной conn
conn = None

try:
    # Попытка подключения
    conn = psycopg2.connect(conn_string)
    cursor = conn.cursor()
    
    # Выполнение тестового запроса
    cursor.execute("SELECT version();")
    print("PostgreSQL version:", cursor.fetchone())

except psycopg2.OperationalError as e:
    # Обработка ошибок подключения
    print(f"Ошибка подключения: {e}")
except Exception as e:
    # Обработка других ошибок
    print(f"Произошла ошибка: {e}")
finally:
    # Закрытие соединения, если оно было установлено
    if conn:
        conn.close()
        print("Соединение закрыто.")

Ошибка подключения: connection to server at "dmprodpostgres2dzc6c48-4597z1.h.ae-rus.net" (10.43.226.38), port 6532 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "dmprodpostgres2dzc6c48-4611z4.h.ae-rus.net" (10.75.226.38), port 6532 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?



In [2]:
import pandas as pd
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv
load_dotenv()

DB_CONFIG = {
    'dbname': os.getenv('DB_NAME'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'host': 'localhost',
    'port': '5432'
}

In [3]:
# Создаем подключение к PostgreSQL
engine = create_engine(f'postgresql+psycopg2://{DB_CONFIG["user"]}:{DB_CONFIG["password"]}@{DB_CONFIG["host"]}:{DB_CONFIG["port"]}/{DB_CONFIG["dbname"]}')

In [4]:
query = """select 
            dt_package
            ,country
            ,service
            ,liter
            ,sum(count_records) as qty
            from public.vw__manifest

            where danger_type = 'General'
            group by 1,2,3,4;"""

In [ ]:
df = pd.read_sql_query(query, engine)

df.to_csv(
    'manifest_report.csv',
    sep=';',            # разделитель - точка с запятой
    index=False,        # не сохранять индекс
    encoding='utf-8',   # кодировка
    #decimal=',',        # десятичный разделитель (если есть числа с плавающей точкой)
    #date_format='%d.%m.%Y'  # формат даты как в исходных данных
)

print(f'Данные успешно сохранены в файл:')
print(f'Количество записей: {len(df)}')

In [7]:
df.shape

(5206, 5)

In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.callbacks import EarlyStopping

ModuleNotFoundError: No module named 'tensorflow'